<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.5：面向对象编程
**上一步：[函数式编程](3.4_functional_programming.ipynb)**<br>
**下一步：[类型](3.6_types.ipynb)**

## 动机
Scala 和 Chisel 都是面向对象的编程语言，这意味着代码可以被划分到对象中。
构建在 Java 之上的 Scala 继承了 Java 的许多面向对象特性。
然而，正如我们将在下面看到的，存在一些差异。
Chisel 的硬件模块类似于 Verilog 的模块，因为它们可以作为单个或多个实例进行实例化和连接。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 面向对象编程
本节概述了 Scala 如何实现面向对象编程范式。到目前为止，您已经了解了类，但 Scala 还具有以下特性：
- [抽象类](#abstract)
- [特质](#traits)
- [对象](#objects)
- [伴生对象](#compobj)
- [样例类](#caseclass)

## 抽象类<a name="abstract"></a>
抽象类与其他编程语言的实现类似。它们可以定义许多未实现的值，子类必须实现这些值。任何对象只能直接从一个父抽象类继承。

<span style="color:blue">**示例：抽象类**</span><br>

In [ ]:
abstract class MyAbstractClass {
  def myFunction(i: Int): Int
  val myValue: String
}
class ConcreteClass extends MyAbstractClass {
  def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}
// 取消注释下面的代码进行测试！
// val abstractClass = new MyAbstractClass() // 非法！无法实例化抽象类
val concreteClass = new ConcreteClass()      // 合法！


## 特质<a name="traits"></a>
特质与抽象类非常相似，因为它们都可以定义未实现的值。但是，它们在两个方面有所不同：
- 一个类可以从多个特质继承
- 特质不能有构造函数参数

<span style="color:blue">**示例：特质和多重继承**</span><br>
特质是 Scala 实现多重继承的方式，如下例所示。`MyClass` 同时扩展了特质 `HasFunction` 和 `HasValue`：

In [ ]:
trait HasFunction {
  def myFunction(i: Int): Int
}
trait HasValue {
  val myValue: String
  val myOtherValue = 100
}
class MyClass extends HasFunction with HasValue {
  override def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}
// 取消注释下面的代码进行测试！
// val myTraitFunction = new HasFunction() // 非法！无法实例化特质
// val myTraitValue = new HasValue()       // 非法！无法实例化特质
val myClass = new MyClass()                // 合法！

要继承多个特质，请像这样链接它们：

```scala
class MyClass extends HasTrait1 with HasTrait2 with HasTrait3 ...
```
通常，除非您确定要强制执行抽象类的单继承限制，否则应始终使用特质而不是抽象类。

## 对象<a name="objects"></a>
Scala 为这些单例类提供了一种语言特性，称为对象。您不能实例化对象 **（无需调用 `new`）**；您可以直接引用它。这使得它们类似于 Java 静态类。

<span style="color:blue">**示例：对象**</span><br>

In [ ]:
object MyObject {
  def hi: String = "Hello World!"
  def apply(msg: String) = msg
}
println(MyObject.hi)
println(MyObject("This message is important!")) // 等同于 MyObject.apply(msg)

## 伴生对象<a name="compobj"></a>

当一个类和一个对象共享相同的名称并在同一个文件中定义时，该对象称为**伴生对象**。当您在类/对象名称前使用 `new` 时，它将实例化该类。如果您不使用 `new`，它将引用该对象：

<span style="color:blue">**示例：伴生对象**</span><br>

In [ ]:
object Lion {
    def roar(): Unit = println("我是一个对象！")
}
class Lion {
    def roar(): Unit = println("我是一个类！")
}
new Lion().roar()
Lion.roar()

伴生对象通常用于以下原因：
  1. 包含与类相关的常量
  2. 在类构造函数之前/之后执行代码
  3. 为类创建多个构造函数

在下面的示例中，我们将实例化多个 Animal 实例。我们希望每只动物都有一个名字，并且知道它在所有实例化中的顺序。最后，如果没有给出名字，它应该得到一个默认名字。

In [ ]:
object Animal {
    val defaultName = "大脚怪"
    private var numberOfAnimals = 0
    def apply(name: String): Animal = {
        numberOfAnimals += 1
        new Animal(name, numberOfAnimals)
    }
    def apply(): Animal = apply(defaultName)
}
class Animal(name: String, order: Int) {
  def info: String = s"你好，我叫 $name，我是第 $order 个！"
}

val bunny = Animal.apply("跳跳") // 调用 Animal 工厂方法
println(bunny.info)
val cat = Animal("晶须")       // 调用 Animal 工厂方法
println(cat.info)
val yeti = Animal()                // 调用 Animal 工厂方法
println(yeti.info)


*这里发生了什么？*
1. 我们的 **Animal 伴生对象** 定义了一个与 ```class Animal``` 相关的常量：
```scala
val defaultName = "大脚怪"
```
2. 它还定义了一个私有可变整数来跟踪 Animal 实例的顺序：
```scala 
private var numberOfAnimals = 0
```
3. 它定义了两个 **apply** 方法，这些方法被称为**工厂方法**，因为它们返回 **class Animal** 的实例。
    1. 第一个方法使用一个参数 ```name``` 创建 Animal 的实例，并使用 ```numberOfAnimals``` 来调用 Animal 类构造函数。
```scala
def apply(name: String): Animal = {
            numberOfAnimals += 1
            new Animal(name, numberOfAnimals)
}
```
    2. 第二个工厂方法不需要参数，而是使用默认名称调用另一个 apply 方法。
```scala
def apply(): Animal = apply(defaultName)
```
4. 这些工厂方法可以像这样简单地调用：
```scala
val bunny = Animal.apply("跳跳")
```
这消除了使用 new 关键字的需要，但真正的魔力在于编译器在看到应用于实例或对象的括号时会假定 apply 方法：
```scala
val cat = Animal("晶须")
```
5. 工厂方法（通常通过伴生对象提供）允许以其他方式表达实例创建，提供对构造函数参数的额外测试、转换，并消除了使用关键字 ```new``` 的需要。请注意，您必须调用伴生对象的 `apply` 方法才能使 `numberOfAnimals` 递增。

**Chisel 使用许多伴生对象，例如 Module。** 当您编写以下代码时：
```scala
val myModule = Module(new MyModule)
```
您正在调用 **Module 伴生对象**，因此 Chisel 可以在实例化 ```MyModule``` 之前和之后运行后台代码。

## 样例类<a name="caseclass"/>
样例类是 Scala 类的一种特殊类型，它提供了一些很酷的附加功能。它们在 Scala 编程中非常常见，因此本节概述了它们的一些有用功能：
- 允许对**类参数**进行**外部访问**
- 实例化类时**无需**使用 **`new`**
- 自动创建一个 **unapply 方法**，该方法提供对所有类参数的访问。
- 不能被子类化

在以下示例中，我们声明了三个不同的类：`Nail`、`Screw` 和 `Staple`。

In [ ]:
class Nail(length: Int) // 普通类
val nail = new Nail(10) // 需要 `new` 关键字
// println(nail.length) // 非法！类构造函数参数默认情况下不可外部访问

class Screw(val threadSpace: Int) // 通过使用 `val` 关键字，threadSpace 现在可以外部访问
val screw = new Screw(2)          // 需要 `new` 关键字
println(screw.threadSpace)

case class Staple(isClosed: Boolean) // 样例类构造函数参数默认情况下可外部访问
val staple = Staple(false)           // 不需要 `new` 关键字
println(staple.isClosed)

`Nail` 是一个普通类，其参数在外部不可见，因为我们没有在参数列表中使用 `val` 关键字。在声明 `Nail` 实例时，它还需要 `new` 关键字。

`Screw` 的声明与 `Nail` 类似，但在参数列表中包含了 `val`。这使得其参数 `threadSpace` 在外部可见。

通过使用样例类，`Staple` 的所有参数都可以在外部可见（无需使用 `val` 关键字）。

此外，在声明样例类时，`Staple` 不需要使用 `new`。这是因为 Scala 编译器会自动为代码中的每个样例类创建一个伴生对象，该对象包含该样例类的 apply 方法。

样例类是包含许多参数的生成器的良好容器。
构造函数为您提供了一个定义派生参数和验证输入的好地方。

In [ ]:
case class SomeGeneratorParameters(
    someWidth: Int,
    someOtherWidth: Int = 10,
    pipelineMe: Boolean = false
) {
    require(someWidth >= 0)
    require(someOtherWidth >= 0)
    val totalWidth = someWidth + someOtherWidth
}

---
# Chisel 中的继承<a name="super"></a>
您之前已经了解过 `Module` 和 `Bundle`，但了解其实际含义非常重要。
您创建的每个 Chisel 模块都是一个扩展基本类型 `Module` 的类。
您创建的每个 Chisel IO 都是一个扩展基本类型 `Bundle`（或者在某些特殊情况下，是 `Bundle` 的超类型 [`Record`](https://github.com/freechipsproject/chisel3/blob/v3.0.0/chiselFrontend/src/main/scala/chisel3/core/Aggregate.scala#L415)）的类。
Chisel 硬件类型（如 `UInt` 或 `Bundle`）都以 `Data` 作为超类型。
我们将探讨使用面向对象编程来创建分层硬件块并探索对象重用。您将在下一个关于类型通用生成器的模块中了解有关类型和 `Data` 的更多信息。

## 模块<a name="module"></a>
每当您想在 Chisel 中创建硬件对象时，它都需要以 `Module` 作为超类。
继承可能并非总是重用的正确工具（[组合优于继承](https://en.wikipedia.org/wiki/Composition_over_inheritance) 是一个常见原则），但继承仍然是一个强大的工具。
下面是一个创建 `Module` 并将它们的多个实例化分层连接在一起的示例。

<span style="color:blue">**示例：格雷码编码器和解码器**</span><br>
我们将创建一个硬件格雷码编码器/解码器。编码或解码操作的选择是硬件可编程的。

In [ ]:
import scala.math.pow

// 创建一个模块
class GrayCoder(bitwidth: Int) extends Module {
  val io = IO(new Bundle{
    val in = Input(UInt(bitwidth.W))
    val out = Output(UInt(bitwidth.W))
    val encode = Input(Bool()) // false 时解码
  })
  
  when (io.encode) { //编码
    io.out := io.in ^ (io.in >> 1.U)
  } .otherwise { // 解码，复杂得多
    io.out := Seq.fill(log2Ceil(bitwidth))(Wire(UInt(bitwidth.W))).zipWithIndex.fold((io.in, 0)){
      case ((w1: UInt, i1: Int), (w2: UInt, i2: Int)) => {
        w2 := w1 ^ (w1 >> pow(2, log2Ceil(bitwidth)-i2-1).toInt.U)
        (w2, i1)
      }
    }._1
  }
}


试试看！

In [ ]:
// 测试我们的格雷码编码器
val bitwidth = 4
test(new GrayCoder(bitwidth)) { c =>
    def toBinary(i: Int, digits: Int = 8) = {
        String.format("%" + digits + "s", i.toBinaryString).replace(' ', '0')
    }
    println("编码：")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(true.B)
        c.clock.step(1)
        println(s"输入 = ${toBinary(i, bitwidth)}, 输出 = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

    println("解码：")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(false.B)
        c.clock.step(1)
        println(s"输入 = ${toBinary(i, bitwidth)}, 输出 = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

}

格雷码通常用于异步接口。通常使用格雷计数器而不是功能齐全的编码器/解码器，但我们将使用上述模块来简化事情。下面是一个 AsyncFIFO 示例，使用上述格雷码编码器构建。控制逻辑和测试器留作后续练习。现在，请查看格雷码编码器是如何多次实例化并连接在一起的。

In [ ]:
class AsyncFIFO(depth: Int = 16) extends Module {
  val io = IO(new Bundle{
    // 写输入
    val write_clock = Input(Clock())
    val write_enable = Input(Bool())
    val write_data = Input(UInt(32.W))
    
    // 读输入/输出
    val read_clock = Input(Clock())
    val read_enable = Input(Bool())
    val read_data = Output(UInt(32.W))
    
    // FIFO 状态
    val full = Output(Bool())
    val empty = Output(Bool())
  })
  
  // 为计数器添加额外的位以检查满/空状态
  assert(isPow2(depth), "AsyncFIFO 需要一个 2 的幂次方的深度！")
  val write_counter = withClock(io.write_clock) { Counter(io.write_enable && !io.full, depth*2)._1 }
  val read_counter = withClock(io.read_clock) { Counter(io.read_enable && !io.empty, depth*2)._1 }
  
  // 编码
  val encoder = new GrayCoder(write_counter.getWidth)
  encoder.io.in := write_counter
  encoder.io.encode := true.B
  
  // 同步
  val sync = withClock(io.read_clock) { ShiftRegister(encoder.io.out, 2) }
  
  // 解码
  val decoder = new GrayCoder(read_counter.getWidth)
  decoder.io.in := sync
  decoder.io.encode := false.B
  
  // 状态逻辑在此处
  
}

---
# 您已完成！

[返回顶部。](#top)